In [1]:
# Restaurant sentences 의 일부
sentences = [
"i want chinese food",
"i want thai food",
"i want italian food",
"i want a restaurant",
"i want to eat",
"i want to eat chinese food",
"i want to eat thai food",
"i am looking for chinese food",
"i am looking for a restaurant",
"can you find a chinese restaurant",
"can you find a thai restaurant",
"can you find an italian restaurant",
"tell me about chinese restaurants",
"tell me about thai restaurants",
"where can i eat chinese food",
]

In [2]:
# Tokenization
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]

# Tokenized sentences를 사용해서 corpus를 생성
corpus = [tokenize(s) for s in sentences]

for sentence in corpus[:3]:
    print(sentence)

['<s>', 'i', 'want', 'chinese', 'food', '</s>']
['<s>', 'i', 'want', 'thai', 'food', '</s>']
['<s>', 'i', 'want', 'italian', 'food', '</s>']


In [3]:
# Unigram counts(코퍼스 내에서 특정 단어가 총 몇 번 등장 했는지) 계산
from collections import Counter

unigram_counts = Counter()

for sentence in corpus:
    unigram_counts.update(sentence)

# 10개의 가장 흔한 단어와 그 빈도를 출력    
print(unigram_counts.most_common(10))

# Unigram 확률 계산

# 단어가 나온 빈도의 합계를 계산
total_words = sum(unigram_counts.values())

def unigram_prob(word):
    # 단어의 unigram 확률을 계산 = (단어의 등장 횟수) / (전체 단어 수)
    return unigram_counts[word] / total_words

# want과 food의 unigram 확률을 계산
print(unigram_prob("i"))
print(unigram_prob("want"))
print(unigram_prob("food"))

[('<s>', 15), ('</s>', 15), ('i', 10), ('want', 7), ('food', 7), ('chinese', 6), ('restaurant', 5), ('thai', 4), ('a', 4), ('eat', 4)]
0.09259259259259259
0.06481481481481481
0.06481481481481481


In [4]:
# Generate restaurant sentences (Unigram)
# Unigram은 이전 단어를 전혀 보지 않음 -> 전체 단어 빈도만으로 다음 단어를 뽑음

import random
random.seed(42)  # 실행할 때마다 같은 결과가 나오도록 시드 고정 (재현성)

def sample_next_word_unigram():
    candidates = []
    weights = []

    # unigram_counts의 모든 단어를 후보로, 등장 횟수를 가중치로 사용 (문맥과 무관)
    for w, count in unigram_counts.items():
        if w == "<s>":  # 시작 토큰은 "다음 단어"가 될 수 없으므로 후보에서 제외
            continue
        candidates.append(w)
        weights.append(count)

    # 후보 단어 중 하나를 빈도 비례로 샘플링하여 반환
    return random.choices(candidates, weights=weights, k=1)[0]

def generate_sentence_unigram(max_length = 15):
    output = []

    for _ in range(max_length):
        next_word = sample_next_word_unigram()

        # 문장 끝
        if next_word == "</s>":
            break

        output.append(next_word)

    return " ".join(output)

for _ in range(10):
    print(generate_sentence_unigram())


restaurant i food chinese am to find i
i chinese thai i chinese to italian chinese a can i can eat
want about
i i you restaurant can am italian restaurants
a you restaurant find a eat i chinese food i chinese i food restaurant

chinese food me to restaurant want am want
where restaurant a eat you for chinese i food food chinese me find food to

tell


In [5]:
# Perplexity Metric (Unigram)

# Perplexity는 언어 모델이 문장을 평가할 때 얼마나 헷갈려하는지를 나타내는 지표로, 낮을수록 모델이 문장을 잘 예측한다는 의미
# 공정한 비교를 위해 Unigram/Bigram/Trigram 모두 "예측해야 하는 토큰" = 문장의 모든 단어 + </s> (N = len(tokens) - 1)로 통일
# (<s>는 항상 주어지는 시작 신호라 예측 대상이 아님)

import math

def perplexity_unigram(sentence):
    tokens = tokenize(sentence)

    log_prob = 0
    targets = tokens[1:]  # <s> 제외, </s> 포함
    for w in targets:
        p = unigram_prob(w)

        # 한 번도 본 적 없는 단어(OOV) -> 확률 0 -> log(0) 불가 -> PP = inf
        if p == 0:
            return float("inf")

        # 부동소수점 연산에서 underflow를 방지하기 위해 log를 사용하여 확률을 더함
        log_prob += math.log(p)

    # log 계산 이후에 지수 함수 사용해서 원상 복구
    return math.exp(-log_prob / len(targets))

# 과제 조건: 테스트 문장은 5개 이상, 훈련 문장은 제외
test_sentences = [
"i want a chinese restaurant",
"where can i eat thai food",
"i love you!!!❤️", # unknown word -> inf 발생!
"my chickdee is so cute💕", # unknown word -> inf 발생!
"can you find thai food"
]

# 훈련 문장이 섞이지 않았는지 코드로 검증
assert not set(test_sentences) & set(sentences), "테스트 문장에 훈련 문장이 포함되어 있음"

for s in test_sentences:
    print(s," : ",  perplexity_unigram(s))


i want a chinese restaurant  :  15.253236287845223
where can i eat thai food  :  22.06985976601904
i love you!!!❤️  :  inf
my chickdee is so cute💕  :  inf
can you find thai food  :  21.71859817351439


In [6]:
# Bigram counts(코퍼스 내에서 특정 bigram이 총 몇 번 등장 했는지) 계산
bigram_counts = Counter()

for sentence in corpus:
    # bigram을 생성하고 bigram_counts에 추가
    # 단어 2개를 보기에 for 문의 범위를 len(sentence) - 1로 설정
    for i in range(len(sentence) - 1):
        # bigram 생성
        bigram = (sentence[i], sentence[i + 1])
        bigram_counts[bigram] += 1 # 바이그램이 등장할 때 마다 카운트 증가 -> 이 조합이 한번 더 나왔네!!

for bigram, count in bigram_counts.most_common(5):
    print(bigram, count)


# Bigram Probability 계산
def bigram_prob(previous_word, word):
    # bigram 확률 = P(word | previous_word) = count(previous_word, word) / count(previous_word) 

    # 이전 단어와 현재 단어의 bigram count
    numerator = bigram_counts[(previous_word, word)]

    # 이전 단어의 unigram count
    denominator = unigram_counts[previous_word]

    # 0으로 나누는 것을 방지하기 위해 조건문 추가
    if denominator == 0:
        return 0
    
    return numerator / denominator

print("\nBigram Probabilities:")
print("P(want | i) =", bigram_prob("i", "want"))
print("P(food | chinese) =", bigram_prob("chinese", "food"))
print("P(restaurant | a) =", bigram_prob("a", "restaurant"))

('<s>', 'i') 9
('i', 'want') 7
('food', '</s>') 7
('restaurant', '</s>') 5
('chinese', 'food') 4

Bigram Probabilities:
P(want | i) = 0.7
P(food | chinese) = 0.6666666666666666
P(restaurant | a) = 0.5


In [7]:
# 'Next Word' Prediction using bigram probabilities
def predict_bigram(previous_word, top_k=3):
    # 이전 단어(previous_word)와 함께 등장하는 단어들의 후보를 찾고, 그 후보들의 확률을 계산하여 상위 top_k개의 후보를 반환
    candidates = []

    for (w1, w2), count in bigram_counts.items():

        # 매개변수로 들어온 이전단어 (previous_word)와 일치하는 경우에만 후보로 추가
        if w1 == previous_word:

            # bigram 확률 계산하고
            probability = bigram_prob(w1, w2)

            # 후보에 넣기
            candidates.append((w2, probability))

    # 확률이 높은 순으로 정렬하고 상위 top_k개의 후보를 반환
    return sorted(candidates, key=lambda x: x[1], reverse=True)[:top_k]

# want 다음에 올 가능성이 가장 높은 상위 3개의 단어와 그 확률을 출력
predict_bigram("want")

[('to', 0.42857142857142855),
 ('chinese', 0.14285714285714285),
 ('thai', 0.14285714285714285)]

In [8]:
# Generate restaurant sentences

import random
random.seed(42)

def sample_next_word_bigram(previous_word):
    candidates = []
    weights = []
    # bigram_counts에서 이전 단어(previous_word)와 함께 등장하는 단어들을 후보로 추가하고, 그 후보들의 count를 weights에 추가
    for (w1, w2), count in bigram_counts.items():
        if w1 == previous_word:
            candidates.append(w2)
            weights.append(count)

    if not candidates:
        return "</s>"

    # 확률 기반 샘플링 -> 다음 단어를 고를 때 무조건 확률이 가장 높은 단어 하나만 고르는 것이 아니라, 확률에 따라 랜덤하게 선택
    return random.choices(candidates,weights=weights,k=1)[0]


def generate_sentence(max_length = 15):
    #    1. "<s>"로 시작
    current = "<s>"
    output = []

    #    2. sample_next_word()를 사용하여 다음 단어를 샘플
    for _ in range(max_length):
        next_word = sample_next_word_bigram(current)
        # 문장 끝
        if next_word == "</s>":
            break
        output.append(next_word)
        # 현재 단어 최신화
        current = next_word

    return " ".join(output)

for _ in range(10):
    print(generate_sentence())


can you find a chinese restaurant
i want chinese food
i want to eat
i am looking for a restaurant
where can you find an italian restaurant
i eat chinese food
can i want to eat
i want thai food
can you find a restaurant
can you find an italian food


In [9]:
# Perplexity Metric

# Perplexity는 언어 모델이 문장을 평가할 때 얼마나 헷갈려하는지를 나타내는 지표로, 낮을수록 모델이 문장을 잘 예측한다는 의미

import math

# Perplexity 계산
def perplexity_bigram(sentence):
    # sentence을 tokenize하기
    tokens = tokenize(sentence)

    log_prob = 0
    n = len(tokens) - 1
    for i in range(n):
        # 토근화한 문장에서 bigram 확률을 계산
        p = bigram_prob(tokens[i],tokens[i + 1])

        # 0으로 나누는 것을 방지하기 위해 조건문 추가
        if p == 0:
         return float("inf")

        # 부동소수점 연산에서 underflow를 방지하기 위해 log를 사용하여 확률을 더함
        log_prob += math.log(p)

    # log 계산 이후에 지수 함수 사용해서 원상 복구
    return math.exp(-log_prob / n)

test_sentences = [
"i want a chinese restaurant",
"where can i eat thai food",
"i love you!!!❤️", # unknown word -> inf 발생!
"my chickdee is so cute💕", # unknown word -> inf 발생!
"can you find thai food"
]

for s in test_sentences:
    print(s," : ",  perplexity_bigram(s))

i want a chinese restaurant  :  2.7144176165949068
where can i eat thai food  :  3.356538286432562
i love you!!!❤️  :  inf
my chickdee is so cute💕  :  inf
can you find thai food  :  inf


In [10]:
# Trigram counts 구하기
# Best practice: Trigram은 문맥이 2단어이므로 문장 앞에 <s>를 2개 붙인다.
#   -> 첫 단어도 P(w1 | <s>, <s>)로 확률이 정의되어 "i"로 고정하지 않고 자연스럽게 문장을 시작할 수 있음

def tokenize_trigram(sentence):
    return ["<s>", "<s>"] + sentence.lower().split() + ["</s>"]

trigram_corpus = [tokenize_trigram(s) for s in sentences]

trigram_counts = Counter()
trigram_context_counts = Counter()  # C(w1, w2) : trigram의 분모 (문맥 2단어 등장 횟수)

for sentence in trigram_corpus:
    for i in range(len(sentence) - 2):
        # 연속된 세 단어를 추출하여 trigram_counts에 추가
        w1, w2, w3 = sentence[i:i+3]
        trigram_counts[(w1, w2, w3)] += 1
        trigram_context_counts[(w1, w2)] += 1


# Trigram Probability 계산 (MLE) : P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2)
def trigram_prob(w1, w2, w3):
    numerator = trigram_counts[(w1, w2, w3)]
    denominator = trigram_context_counts[(w1, w2)]

    # 0으로 나누는 것을 방지하기 위해 조건문 추가
    if denominator == 0:
        return 0

    return numerator / denominator

# i와 want 다음에 chinese가 올 확률을 계산
print("P(chinese | i, want) =", trigram_prob("i", "want", "chinese"))
# 문장 첫 단어 확률도 정의됨
print("P(i | <s>, <s>) =", trigram_prob("<s>", "<s>", "i"))


P(chinese | i, want) = 0.14285714285714285
P(i | <s>, <s>) = 0.6


In [11]:
# Generate restaurant sentences (Trigram)

import random
random.seed(42)

def sample_next_word_trigram(w1, w2):
    # 이전 두 단어 (w1, w2)를 문맥으로 다음 단어를 샘플링 -> 트라이그램 모델의 핵심
    candidates = []
    weights = []
    for (a, b, c), count in trigram_counts.items():
        if a == w1 and b == w2:
            candidates.append(c)
            weights.append(count)

    if not candidates:
        return "</s>"

    # 확률 기반 샘플링 -> 무조건 최댓값이 아니라 확률에 따라 랜덤하게 선택
    return random.choices(candidates, weights=weights, k=1)[0]


def generate_sentence_trigram(max_length = 15):
    # 1. (<s>, <s>)로 시작 -> 첫 단어도 학습된 분포 P(w | <s>, <s>)에서 뽑음
    #    (이전 버전은 두 번째 단어를 "i"로 고정해서 "can you ...", "tell me ..." 같은 문장을 절대 만들 수 없었음)
    w1, w2 = "<s>", "<s>"
    output = []

    # 2. sample_next_word_trigram()으로 다음 단어를 샘플
    for _ in range(max_length):
        next_word = sample_next_word_trigram(w1, w2)

        # 문장 끝
        if next_word == "</s>":
            break

        output.append(next_word)
        # 문맥 한 칸 이동
        w1, w2 = w2, next_word

    return " ".join(output)

# 과제 조건: 5개 문장 출력
for _ in range(5):
    print(generate_sentence_trigram())


can you find a thai restaurant
i want chinese food
i want to eat
i am looking for a restaurant
where can i eat chinese food


In [12]:
# Perplexity Metric (Trigram, MLE)

import math

def perplexity_trigram(sentence):
    tokens = tokenize_trigram(sentence)  # <s> <s> w1 ... wn </s>

    log_prob = 0
    n = len(tokens) - 2  # 예측 대상 = w1 ... wn, </s>  (Unigram/Bigram과 같은 개수)
    for i in range(n):
        p = trigram_prob(tokens[i], tokens[i + 1], tokens[i + 2])

        # 학습 데이터에 없는 trigram -> 확률 0 -> PP = inf (데이터 희소성 문제)
        if p == 0:
            return float("inf")

        log_prob += math.log(p)

    return math.exp(-log_prob / n)

for s in test_sentences:
    print(s," : ",  perplexity_trigram(s))


i want a chinese restaurant  :  inf
where can i eat thai food  :  inf
i love you!!!❤️  :  inf
my chickdee is so cute💕  :  inf
can you find thai food  :  inf


## 왜 Trigram(MLE)은 테스트 문장에서 전부 inf가 나올까?

- 코퍼스가 15문장뿐이라 테스트 문장의 3단어 조합 중 하나라도 학습 때 본 적이 없으면 확률이 0이 되고, perplexity는 무한대가 된다.
- 즉 n이 커질수록 문맥은 정교해지지만 **데이터 희소성(sparsity)** 이 심해진다.
- 해결책 중 하나가 강의 흐름에 나온 **Interpolation**: 세 모델의 확률을 가중합해서 0 확률을 피한다.

  P(w3 | w1, w2) = λ3·P_tri + λ2·P_bi + λ1·P_uni  (λ1 + λ2 + λ3 = 1)

- 단, 학습 때 한 번도 보지 못한 단어(OOV)는 unigram 확률도 0이라 여전히 inf → 실무에서는 `<unk>` 토큰으로 처리한다.

In [13]:
# Interpolation으로 Unigram / Bigram / Trigram 비교

def interpolated_prob(w1, w2, w3, lambdas=(0.1, 0.3, 0.6)):
    l1, l2, l3 = lambdas
    # 문장 맨 앞 (<s>, <s>) 문맥에서 bigram은 P(w3 | <s>)로 계산
    return (l3 * trigram_prob(w1, w2, w3)
            + l2 * bigram_prob(w2, w3)
            + l1 * unigram_prob(w3))

def perplexity_interpolated(sentence, lambdas=(0.1, 0.3, 0.6)):
    tokens = tokenize_trigram(sentence)
    log_prob = 0
    n = len(tokens) - 2
    for i in range(n):
        p = interpolated_prob(tokens[i], tokens[i + 1], tokens[i + 2], lambdas)
        if p == 0:
            return float("inf")
        log_prob += math.log(p)
    return math.exp(-log_prob / n)

def fmt(x):
    return "inf" if math.isinf(x) else f"{x:.2f}"

print(f"{'test sentence':<28} {'Unigram':>8} {'Bigram':>8} {'Trigram':>8} {'Interp':>8}")
print("-" * 64)
for s in test_sentences:
    print(f"{s:<28} {fmt(perplexity_unigram(s)):>8} {fmt(perplexity_bigram(s)):>8} "
          f"{fmt(perplexity_trigram(s)):>8} {fmt(perplexity_interpolated(s)):>8}")


test sentence                 Unigram   Bigram  Trigram   Interp
----------------------------------------------------------------
i want a chinese restaurant     15.25     2.71      inf     2.73
where can i eat thai food       22.07     3.36      inf     2.58
i love you!!!❤️                   inf      inf      inf      inf
my chickdee is so cute💕           inf      inf      inf      inf
can you find thai food          21.72      inf      inf     4.89
